# Weather Data Analysis Project
**Course:** _<your course name>_  |  **Student:** _<your name, roll no.>_  |  **Date:** _<date>_

This notebook analyses historical daily weather data for a city and covers:
1. Data collection (Open-Meteo Historical Weather API, free, no API key needed)
2. Data cleaning and feature engineering
3. Exploratory data analysis (trends, seasonality, distributions)
4. Climate trend and extreme-event analysis
5. Correlation analysis
6. Machine-learning model to predict next-day mean temperature
7. Conclusions

**How to run in Google Colab:** upload this file, then click *Runtime > Run all*. Change the city in Section 1 to analyse another location.

## 1. Setup and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)

# ---------- CONFIGURATION: change these for your own city ----------
CITY = "Delhi"
LATITUDE, LONGITUDE = 28.61, 77.21
START_DATE, END_DATE = "2014-01-01", "2023-12-31"
# --------------------------------------------------------------------
print(f"Analysing {CITY} from {START_DATE} to {END_DATE}")

## 2. Data collection
Daily data is downloaded from the Open-Meteo archive API. If the download fails (no internet, API down), the notebook falls back to a **synthetic demo dataset** so that the code still runs. A warning is printed when that happens. **Do not report synthetic results as real findings.**

Alternatively, set `CSV_PATH` to your own CSV (it needs a `date` column plus the weather columns below).

In [ ]:
CSV_PATH = None   # e.g. "/content/my_weather.csv" to use your own file

DAILY_VARS = [
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "precipitation_sum", "wind_speed_10m_max", "shortwave_radiation_sum",
]

def fetch_open_meteo():
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": LATITUDE, "longitude": LONGITUDE,
        "start_date": START_DATE, "end_date": END_DATE,
        "daily": ",".join(DAILY_VARS), "timezone": "auto",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    d = r.json()["daily"]
    df = pd.DataFrame(d).rename(columns={"time": "date"})
    return df

def make_synthetic():
    rng = np.random.default_rng(42)
    dates = pd.date_range(START_DATE, END_DATE, freq="D")
    doy = dates.dayofyear.values
    t = np.arange(len(dates))
    mean = 25 + 10 * np.sin(2 * np.pi * (doy - 105) / 365) + 0.0004 * t + rng.normal(0, 1.8, len(dates))
    monsoon = ((doy > 180) & (doy < 270)).astype(float)
    rain = rng.gamma(0.6, 8, len(dates)) * (rng.random(len(dates)) < (0.1 + 0.55 * monsoon))
    df = pd.DataFrame({
        "date": dates,
        "temperature_2m_mean": mean,
        "temperature_2m_max": mean + 5 + rng.normal(0, 1, len(dates)),
        "temperature_2m_min": mean - 5 + rng.normal(0, 1, len(dates)),
        "precipitation_sum": rain,
        "wind_speed_10m_max": np.abs(rng.normal(14, 5, len(dates))),
        "shortwave_radiation_sum": np.clip(20 + 8 * np.sin(2 * np.pi * (doy - 80) / 365) - 0.2 * rain + rng.normal(0, 2, len(dates)), 0, None),
    })
    return df

SOURCE = None
try:
    if CSV_PATH:
        df_raw = pd.read_csv(CSV_PATH); SOURCE = "user CSV"
    else:
        df_raw = fetch_open_meteo(); SOURCE = "Open-Meteo API"
except Exception as e:
    print("WARNING: could not load real data ->", e)
    print("WARNING: using SYNTHETIC demo data. Do not report these results as real.")
    df_raw = make_synthetic(); SOURCE = "SYNTHETIC demo data"

df_raw["date"] = pd.to_datetime(df_raw["date"])
print("Data source:", SOURCE)
print("Shape:", df_raw.shape)
df_raw.head()

## 3. Data cleaning and understanding

In [ ]:
df = df_raw.copy().sort_values("date").drop_duplicates("date").set_index("date")

print("=== Data types ===")
print(df.dtypes, "\n")
print("=== Missing values per column ===")
print(df.isna().sum(), "\n")
print("=== Summary statistics ===")
df.describe().round(2)

In [ ]:
# Fill small gaps by time interpolation, then check
n_missing_before = int(df.isna().sum().sum())
df = df.interpolate(method="time", limit=5).dropna()
print(f"Missing cells before: {n_missing_before} | after cleaning: {int(df.isna().sum().sum())}")

# Outlier check using the IQR rule (reported, not removed: extreme weather is real data)
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

print("\nIQR outlier counts:")
print({c: iqr_outliers(df[c]) for c in df.columns})

## 4. Feature engineering

In [ ]:
def season_india(m):
    if m in (12, 1, 2):  return "Winter"
    if m in (3, 4, 5):   return "Summer"
    if m in (6, 7, 8, 9): return "Monsoon"
    return "Post-monsoon"

df["year"] = df.index.year
df["month"] = df.index.month
df["month_name"] = df.index.strftime("%b")
df["season"] = df["month"].map(season_india)
df["temp_range"] = df["temperature_2m_max"] - df["temperature_2m_min"]
df["rainy_day"] = (df["precipitation_sum"] >= 2.5).astype(int)   # >= 2.5 mm counts as a rainy day
df.head()

## 5. Exploratory data analysis

### 5.1 Temperature over time

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["temperature_2m_mean"], lw=0.5, alpha=0.5, label="Daily mean")
ax.plot(df.index, df["temperature_2m_mean"].rolling(30, center=True).mean(), lw=2, color="crimson", label="30-day rolling mean")
ax.set(title=f"Daily mean temperature, {CITY}", xlabel="Date", ylabel="Temperature (°C)")
ax.legend(); plt.tight_layout(); plt.savefig("fig_temperature_timeseries.png", dpi=150); plt.show()

### 5.2 Seasonality: monthly climatology

In [ ]:
order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
monthly = df.groupby("month_name").agg(
    mean_temp=("temperature_2m_mean", "mean"),
    max_temp=("temperature_2m_max", "mean"),
    min_temp=("temperature_2m_min", "mean"),
    total_rain_per_year=("precipitation_sum", lambda s: s.sum() / df["year"].nunique()),
).reindex(order).round(2)
display(monthly)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
monthly[["max_temp", "mean_temp", "min_temp"]].plot(ax=axes[0], marker="o")
axes[0].set(title="Average monthly temperature", ylabel="°C", xlabel="")
monthly["total_rain_per_year"].plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set(title="Average monthly rainfall", ylabel="mm per year", xlabel="")
plt.tight_layout(); plt.savefig("fig_monthly_climatology.png", dpi=150); plt.show()

### 5.3 Distributions by month and season

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="month_name", y="temperature_2m_mean", order=order, ax=axes[0], color="skyblue")
axes[0].set(title="Mean temperature by month", xlabel="", ylabel="°C")
sns.violinplot(data=df, x="season", y="temperature_2m_mean",
               order=["Winter", "Summer", "Monsoon", "Post-monsoon"], ax=axes[1], palette="Set2")
axes[1].set(title="Mean temperature by season", xlabel="", ylabel="°C")
plt.tight_layout(); plt.savefig("fig_distributions.png", dpi=150); plt.show()

### 5.4 Rainfall analysis

In [ ]:
annual_rain = df.groupby("year")["precipitation_sum"].sum()
rainy_days = df.groupby("year")["rainy_day"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
annual_rain.plot(kind="bar", ax=axes[0], color="teal")
axes[0].set(title="Total annual rainfall", ylabel="mm", xlabel="Year")
rainy_days.plot(kind="bar", ax=axes[1], color="slateblue")
axes[1].set(title="Number of rainy days per year (>= 2.5 mm)", ylabel="days", xlabel="Year")
plt.tight_layout(); plt.savefig("fig_rainfall.png", dpi=150); plt.show()
print(f"Wettest year: {annual_rain.idxmax()} ({annual_rain.max():.0f} mm) | Driest year: {annual_rain.idxmin()} ({annual_rain.min():.0f} mm)")

## 6. Climate trend analysis

In [ ]:
yearly = df.groupby("year")["temperature_2m_mean"].mean()
if len(yearly) > 0 and df.index.year.nunique() > 2:
    # exclude incomplete years from the trend
    counts = df.groupby("year").size()
    yearly = yearly[counts >= 350]

slope, intercept, r, p, se = stats.linregress(yearly.index, yearly.values)
print(f"Trend: {slope:+.3f} °C per year  ({slope*10:+.2f} °C per decade)")
print(f"R² = {r**2:.3f},  p-value = {p:.4f}  ->  {'statistically significant' if p < 0.05 else 'not statistically significant'} at the 5% level")

fig, ax = plt.subplots()
ax.plot(yearly.index, yearly.values, "o-", label="Annual mean temperature")
ax.plot(yearly.index, intercept + slope * yearly.index, "r--", label=f"Linear trend ({slope*10:+.2f} °C/decade)")
ax.set(title=f"Annual mean temperature trend, {CITY}", xlabel="Year", ylabel="°C"); ax.legend()
plt.tight_layout(); plt.savefig("fig_trend.png", dpi=150); plt.show()

## 7. Extreme weather events

In [ ]:
hot_thr = df["temperature_2m_max"].quantile(0.95)
cold_thr = df["temperature_2m_min"].quantile(0.05)
print(f"Hot day threshold (95th percentile of max temp): {hot_thr:.1f} °C")
print(f"Cold day threshold (5th percentile of min temp): {cold_thr:.1f} °C")

df["hot_day"] = df["temperature_2m_max"] >= hot_thr
df["cold_day"] = df["temperature_2m_min"] <= cold_thr

# Heatwave = at least 3 consecutive hot days
grp = (df["hot_day"] != df["hot_day"].shift()).cumsum()
runs = df.groupby(grp)["hot_day"].agg(["first", "size"])
heatwaves = runs[(runs["first"]) & (runs["size"] >= 3)]
print(f"Heatwave spells (>= 3 consecutive hot days): {len(heatwaves)}")

print("\nTop 5 hottest days:")
display(df.nlargest(5, "temperature_2m_max")[["temperature_2m_max", "temperature_2m_min", "precipitation_sum"]])
print("Top 5 heaviest rainfall days:")
display(df.nlargest(5, "precipitation_sum")[["precipitation_sum", "temperature_2m_mean"]])

df.groupby("year")[["hot_day", "cold_day"]].sum().plot(kind="bar", figsize=(11, 4), title="Hot and cold days per year")
plt.ylabel("days"); plt.tight_layout(); plt.savefig("fig_extremes.png", dpi=150); plt.show()

## 8. Correlation analysis

In [ ]:
num_cols = ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
            "precipitation_sum", "wind_speed_10m_max", "shortwave_radiation_sum", "temp_range"]
corr = df[num_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation matrix of weather variables")
plt.tight_layout(); plt.savefig("fig_correlation.png", dpi=150); plt.show()

## 9. Predictive modelling: next-day mean temperature
**Goal:** predict tomorrow's mean temperature from today's and recent weather.
**Features:** current-day weather, temperature lags (1, 2, 3, 7 days), 7-day rolling mean, and seasonal encoding (sin/cos of day of year).
**Validation:** chronological split (first 80% train, last 20% test). Random shuffling would leak future information, so it is not used for time series.

In [ ]:
m = df[["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
        "precipitation_sum", "wind_speed_10m_max", "shortwave_radiation_sum"]].copy()
for lag in (1, 2, 3, 7):
    m[f"temp_lag{lag}"] = m["temperature_2m_mean"].shift(lag)
m["temp_roll7"] = m["temperature_2m_mean"].rolling(7).mean()
doy = m.index.dayofyear
m["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
m["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
m["target"] = m["temperature_2m_mean"].shift(-1)       # tomorrow's mean temperature
m = m.dropna()

X, y = m.drop(columns="target"), m["target"]
split = int(len(m) * 0.8)
X_train, X_test, y_train, y_test = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]
print(f"Train: {len(X_train)} days | Test: {len(X_test)} days")

# Baseline: "tomorrow = today" (persistence)
base_pred = X_test["temperature_2m_mean"]
models = {
    "Persistence baseline": None,
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1),
}
results, preds = [], {}
for name, mdl in models.items():
    if mdl is None:
        p = base_pred
    else:
        mdl.fit(X_train, y_train); p = mdl.predict(X_test)
    preds[name] = p
    results.append({"Model": name,
                    "MAE (°C)": mean_absolute_error(y_test, p),
                    "RMSE (°C)": np.sqrt(mean_squared_error(y_test, p)),
                    "R²": r2_score(y_test, p)})
results_df = pd.DataFrame(results).round(3)
results_df

In [ ]:
best = results_df.sort_values("RMSE (°C)").iloc[0]["Model"]
print("Best model by RMSE:", best)

fig, ax = plt.subplots()
ax.plot(y_test.index[:120], y_test.values[:120], label="Actual", lw=2)
ax.plot(y_test.index[:120], np.asarray(preds["Random Forest"])[:120], label="Random Forest", lw=1.5)
ax.plot(y_test.index[:120], np.asarray(preds["Linear Regression"])[:120], label="Linear Regression", lw=1.5, alpha=0.8)
ax.set(title="Actual vs predicted next-day mean temperature (first 120 test days)", ylabel="°C"); ax.legend()
plt.tight_layout(); plt.savefig("fig_predictions.png", dpi=150); plt.show()

rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh", figsize=(8, 6), title="Random Forest feature importance", color="darkorange")
plt.tight_layout(); plt.savefig("fig_feature_importance.png", dpi=150); plt.show()

## 10. Save outputs

In [ ]:
df.to_csv(f"{CITY.lower()}_weather_clean.csv")
results_df.to_csv("model_results.csv", index=False)
print("Saved: cleaned dataset, model_results.csv and all fig_*.png charts.")

# In Colab, uncomment to download everything as a zip:
# !zip -q outputs.zip *.png *.csv
# from google.colab import files; files.download("outputs.zip")

## 11. Conclusions
Fill this in after running the notebook with **real data** (check that `Data source` above does not say SYNTHETIC):

1. **Seasonality:** which months are hottest/coldest, and when does most rainfall occur?
2. **Trend:** what is the temperature trend per decade, and is it statistically significant?
3. **Extremes:** how many hot/cold days and heatwave spells occurred, and are they becoming more frequent?
4. **Relationships:** which variables are most strongly correlated?
5. **Prediction:** which model performed best, and how much better is it than the persistence baseline?

**Limitations:** reanalysis-based gridded data (not a single station reading), a limited 10-year window, and a simple model without atmospheric pressure or humidity inputs.

**Data source:** Open-Meteo Historical Weather API (https://open-meteo.com/en/docs/historical-weather-api).